# plotly: gráficos interactivos

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gf0657-programacionsig/2026-ii/blob/main/contenidos/iii-analisis-visualizacion-datos/15-plotly-graficos-interactivos.ipynb)

## Trabajo previo

### Lecturas

Antes de la clase, revise las siguientes páginas de la documentación de plotly, en inglés. La primera presenta plotly express, el módulo que se usa en este cuaderno, con ejemplos de cada tipo de gráfico; la segunda explica cómo personalizar los gráficos producidos.

Plotly. (s. f.-a). Plotly Express in Python. En *Plotly open source graphing library for Python*. Recuperado el 26 de setiembre de 2026, de https://plotly.com/python/plotly-express/
\
\
Plotly. (s. f.-b). Styling Plotly Express figures in Python. En *Plotly open source graphing library for Python*. Recuperado el 26 de setiembre de 2026, de https://plotly.com/python/styling-plotly-express/

### Otros recursos

La [documentación de plotly para Python](https://plotly.com/python/) tiene un tutorial por cada tipo de gráfico ([barras](https://plotly.com/python/bar-charts/), [líneas](https://plotly.com/python/line-charts/), [histogramas](https://plotly.com/python/histograms/), [dispersión](https://plotly.com/python/line-and-scatter/), [cajas](https://plotly.com/python/box-plots/)) y la [referencia de plotly express](https://plotly.com/python-api-reference/plotly.express.html) documenta los argumentos de cada función.

## Introducción

Este cuaderno cierra la sección de análisis y visualización de datos. Los gráficos de la [parte III](https://gf0657-programacionsig.github.io/2026-ii/pandas-graficos-matplotlib/) de la serie *pandas*, hechos con matplotlib, son **estáticos**: imágenes, adecuadas para documentos, presentaciones e informes impresos. Los gráficos de este cuaderno son **interactivos**: al pasar el cursor sobre un punto o una barra aparece su información, se puede acercar una región, desplazarse por ella, ocultar series con un clic en la leyenda y descargar la imagen. Esa interactividad es valiosa para **explorar** datos con muchas observaciones (¿cuál es ese cantón que se sale del patrón?) y es la norma en las **aplicaciones web**, como las que se desarrollarán con streamlit al final del curso.

[plotly](https://plotly.com/python/) es la biblioteca de gráficos interactivos más usada en Python. Sus gráficos están hechos, en el fondo, con [JavaScript](https://es.wikipedia.org/wiki/JavaScript), el lenguaje de los navegadores web, por lo que funcionan en cualquier página: en un cuaderno de notas, en el sitio del curso o en una aplicación. Su módulo de alto nivel, **plotly express** (`px`), produce un gráfico completo con una función por tipo de gráfico, a partir de un DataFrame y de los nombres de sus columnas: el mismo enfoque de `plot()` de pandas, con más resultados por línea de código.

El cuaderno sigue el orden de la parte III (barras, líneas, histogramas, dispersión), agrega los **gráficos de caja** y las **facetas**, y termina con la comparación entre las dos bibliotecas: cuándo conviene cada una. Los datos son los mismos: la [Estimación de Población y Vivienda 2022](https://inec.cr/) del INEC, la serie de censos y los casos de COVID-19 por cantón del Ministerio de Salud.

Como todos los cuadernos del curso, puede ejecutarse en la nube, con la insignia "Abrir en Google Colab", o localmente, en [Visual Studio Code](https://gf0657-programacionsig.github.io/2026-ii/vscode/) con el ambiente `geopython` como kernel. El archivo `.ipynb` puede descargarse con el botón de descarga (ícono de flecha hacia abajo) en la parte superior de la página del cuaderno, en el sitio web del curso.

In [1]:
import pandas as pd
import plotly.express as px
from IPython.display import display

# Función auxiliar del curso: despliega un gráfico de plotly de forma compatible con el sitio web,
# Jupyter y Colab. Se usa en lugar de fig.show() (ver la sección sobre fig.show() en la clínica de errores).
def mostrar(fig):
    display(fig)

# Direcciones de los directorios de datos en el repositorio del curso
DATOS = "https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/datos/inec"
DATOS_SALUD = "https://raw.githubusercontent.com/gf0657-programacionsig/2026-ii/main/datos/ministerio-salud"

# Los mismos datos de la parte III de la serie pandas
provincias = pd.read_csv(DATOS + "/provincias-2022.csv")
cantones = pd.read_csv(DATOS + "/cantones-2022.csv")
distritos = pd.read_csv(DATOS + "/distritos-2022.csv")

censos = pd.read_excel(
    DATOS + "/reResultadosEstimacionPoblacionVivienda2022_3.xlsx",
    sheet_name="1", skiprows=6, nrows=11, header=None,
    names=["anio", "poblacion", "hombres", "mujeres", "tasa_crecimiento"],
    na_values="-"
)

covid = pd.read_csv(
    DATOS_SALUD + "/05_30_22_CSV_POSITIVOS.csv",
    sep=";", encoding="cp1252", usecols=["cod_canton", "30/05/2022"]
).dropna()
covid = covid.rename(columns={"cod_canton": "codigo_canton", "30/05/2022": "positivos"}).astype(int)
cantones_covid = cantones.merge(covid, on="codigo_canton", how="left")
cantones_covid["tasa_100k"] = (cantones_covid["positivos"] / cantones_covid["poblacion"] * 100000).round(0)

# Configuración por defecto de plotly express para todo el cuaderno:
# fondo blanco y los dos colores de la parte III
AZUL = "#1F77B4"
NARANJA = "#D97B1B"
px.defaults.template = "plotly_white"
px.defaults.color_discrete_sequence = [AZUL, NARANJA, "#2CA02C", "#9467BD", "#8C564B", "#E377C2", "#7F7F7F"]

print(provincias.shape, cantones.shape, distritos.shape, censos.shape, cantones_covid.shape)

(7, 14) (82, 16) (487, 17) (11, 5) (82, 18)


## Un primer gráfico

Cada función de plotly express recibe un DataFrame y los nombres de las columnas que van en cada eje, y retorna una **figura**. El primer gráfico de la parte III, con `px.bar()`:

In [2]:
# Población de las provincias, en barras
fig = px.bar(provincias.sort_values("poblacion", ascending=False), x="provincia", y="poblacion")

mostrar(fig)

Antes de seguir, interactúe con el gráfico:

- Pase el cursor sobre una barra: aparece una **etiqueta emergente** (*hover*) con la provincia y su población exacta.
- Arrastre el cursor sobre una región para **acercarla** (*zoom*); haga doble clic para volver a la vista completa.
- En la esquina superior derecha aparece una **barra de herramientas** con acercar, alejar, desplazar (*pan*) y descargar el gráfico como imagen PNG.

Aun así, el gráfico tiene los mismos defectos de comunicación del primer gráfico de la parte III: sin título, con los nombres de las columnas como etiquetas de los ejes y sin separador de miles. En plotly express, casi todo se corrige con argumentos de la misma función:

- `title`: el título.
- `labels`: un diccionario que traduce los nombres de las columnas a etiquetas legibles, en los ejes **y** en las etiquetas emergentes.
- `text_auto`: escribe el valor sobre cada barra, con un formato opcional.

Y lo que no es un argumento se ajusta después con los métodos `update_layout()` (título, ejes, leyenda, tamaño: la disposición general) y `update_traces()` (color, tamaño y forma de las marcas: los datos dibujados):

In [3]:
# El mismo gráfico, para comunicar
fig = px.bar(
    provincias.sort_values("poblacion", ascending=False),
    x="provincia",
    y="poblacion",
    title="Población de las provincias de Costa Rica, 2022",
    labels={"provincia": "Provincia", "poblacion": "Habitantes"},
    text_auto=",.0f"  # valor sobre cada barra, con separador de miles
)

fig.update_layout(xaxis_title="", yaxis_tickformat=",")  # sin etiqueta en x; miles en el eje y

mostrar(fig)

Los formatos de número de plotly (`",.0f"`, `","`) siguen la [sintaxis de d3](https://github.com/d3/d3-format#locale_format), muy parecida a la de las hileras de Python: la coma es el separador de miles y `.0f`, cero decimales.

Una figura de plotly se compone de **trazas** (`fig.data`: cada serie de datos dibujada, aquí una traza de barras) y de una **disposición** (`fig.layout`: título, ejes, leyenda, tamaño, fondo). `update_traces()` modifica las primeras; `update_layout()`, la segunda. Es la misma división que la de los datos y los `Axes` en matplotlib.

## Tipos de gráficos

### Barras

Los argumentos de `px.bar()` cubren las variantes de la parte III. `orientation="h"` produce barras horizontales (con `x` e `y` intercambiados); como en pandas, la primera fila queda abajo:

In [4]:
# Los diez cantones más poblados, en barras horizontales
top_10 = cantones.sort_values("poblacion", ascending=False).head(10)

fig = px.bar(
    top_10.sort_values("poblacion"),
    x="poblacion",
    y="canton",
    orientation="h",
    title="Los diez cantones más poblados de Costa Rica, 2022",
    labels={"poblacion": "Habitantes", "canton": "Cantón", "provincia": "Provincia"},
    hover_data=["provincia"],  # columna adicional en la etiqueta emergente
    text_auto=",.0f"
)

fig.update_layout(yaxis_title="", xaxis_tickformat=",")

mostrar(fig)

`hover_data` agrega columnas a la etiqueta emergente: aquí, la provincia de cada cantón, un dato que en el gráfico estático no cabía. Para **varias series**, `y` recibe una lista de columnas y `barmode="group"` las coloca lado a lado (sin `barmode`, plotly las apila):

In [5]:
# Población por sexo de cada provincia: barras agrupadas
fig = px.bar(
    provincias.sort_values("poblacion", ascending=False),
    x="provincia",
    y=["hombres", "mujeres"],
    barmode="group",
    title="Población por sexo de las provincias de Costa Rica, 2022",
    labels={"provincia": "Provincia", "value": "Habitantes", "variable": "Sexo"}
)

fig.update_layout(xaxis_title="", yaxis_tickformat=",", legend_title_text="")

# Nombres de las series en la leyenda, en lugar de los nombres de las columnas
fig.for_each_trace(lambda traza: traza.update(name=traza.name.capitalize()))

mostrar(fig)

Cuando `y` es una lista, plotly express nombra internamente las columnas `variable` (la serie) y `value` (el valor), y esos son los nombres que se traducen en `labels`. Haga clic en "Mujeres" en la leyenda: la serie se oculta y el gráfico se reajusta; otro clic la restaura.

### Líneas

`px.line()` produce gráficos de líneas; `markers=True` agrega los marcadores. Con una lista en `y`, dibuja una línea por columna, cada una con su color y su entrada en la leyenda:

In [6]:
# Población por sexo de Costa Rica en los censos, 1864-2022
fig = px.line(
    censos,
    x="anio",
    y=["hombres", "mujeres"],
    markers=True,
    title="Población por sexo de Costa Rica en los censos, 1864-2022",
    labels={"anio": "Año del censo", "value": "Habitantes", "variable": "Sexo"}
)

fig.update_layout(yaxis_tickformat=",", legend_title_text="")
fig.for_each_trace(lambda traza: traza.update(name=traza.name.capitalize()))

mostrar(fig)

Este gráfico ilustra una ventaja de la interactividad: en la versión estática, las dos líneas se superponen y no se distingue en qué censos hubo más mujeres que hombres; aquí, el cursor sobre cada censo muestra los dos valores exactos, y al acercar el período 1864-1950 se ven los cruces.

### Histogramas

`px.histogram()` produce histogramas; `nbins` es la cantidad aproximada de intervalos. La etiqueta emergente de cada barra muestra el intervalo y el conteo, que en matplotlib había que leer del eje:

In [7]:
# Distribución de la población de los distritos
fig = px.histogram(
    distritos,
    x="poblacion",
    nbins=40,
    title="Distribución de la población de los distritos de Costa Rica, 2022",
    labels={"poblacion": "Habitantes del distrito"}
)

fig.update_layout(yaxis_title="Cantidad de distritos", xaxis_tickformat=",", bargap=0.05)

mostrar(fig)

### Dispersión

`px.scatter()` produce gráficos de dispersión, y es donde plotly express más aporta: además de `x` e `y`, otras variables pueden codificarse en el **color** de los puntos (`color`), en su **tamaño** (`size`) y en la etiqueta emergente (`hover_name`, `hover_data`). El gráfico de densidad y casos de COVID-19 de la parte III, ahora con la provincia en el color, la población en el tamaño y el nombre del cantón al pasar el cursor:

In [8]:
# Densidad de población y tasa de casos de COVID-19 por cantón
fig = px.scatter(
    cantones_covid,
    x="densidad",
    y="tasa_100k",
    color="provincia",
    size="poblacion",
    hover_name="canton",
    hover_data={"positivos": ":,", "poblacion": ":,", "densidad": ":,.1f", "tasa_100k": ":,.0f"},
    log_x=True,
    title="Densidad de población y casos de COVID-19 en los cantones, al 30-05-2022",
    labels={
        "densidad": "Densidad de población (habitantes por km²)",
        "tasa_100k": "Casos positivos por 100 000 habitantes",
        "provincia": "Provincia",
        "poblacion": "Población",
        "positivos": "Casos positivos"
    }
)

fig.update_layout(yaxis_tickformat=",", height=500)

mostrar(fig)

Tres detalles:

- `log_x=True` es la escala logarítmica del eje horizontal, igual que `set_xscale("log")` en matplotlib.
- En `hover_data`, un diccionario permite indicar el **formato** de cada columna en la etiqueta emergente (`":,"`: separador de miles), además de cuáles se muestran.
- La leyenda por provincia es interactiva: un clic oculta una provincia; un doble clic la deja sola. Así se explora, por ejemplo, si los cantones de Guanacaste siguen el patrón general.

El gráfico codifica cuatro variables (densidad, tasa, provincia y población) más el nombre. Es tentador agregar más; el criterio sigue siendo el de la parte III: cada codificación debe ayudar a responder la pregunta, no solo ser posible.

### Gráficos de caja

Un [gráfico de caja](https://es.wikipedia.org/wiki/Diagrama_de_caja) (*box plot*) resume la **distribución de una variable numérica en cada categoría**: la caja va del primer al tercer cuartil (la mitad central de los datos), la línea interior es la mediana, los "bigotes" abarcan el resto de los valores y los puntos aislados son valores atípicos. Es el gráfico para comparar distribuciones entre grupos, algo que un histograma por grupo haría con mucho más espacio. La densidad de los cantones de cada provincia, con `points="all"` para ver además cada cantón:

In [9]:
# Distribución de la densidad de población de los cantones, por provincia
fig = px.box(
    cantones,
    x="provincia",
    y="densidad",
    points="all",  # además de la caja, un punto por cantón
    hover_name="canton",
    log_y=True,
    title="Densidad de población de los cantones, por provincia, 2022",
    labels={"provincia": "Provincia", "densidad": "Habitantes por km² (escala logarítmica)"}
)

fig.update_layout(xaxis_title="")

mostrar(fig)

San José y Heredia tienen las medianas más altas y las cajas más largas: sus cantones van de los muy densos del área metropolitana a los rurales (Pérez Zeledón, Sarapiquí). Guanacaste tiene la mediana más baja y la caja más compacta. La escala logarítmica vuelve a ser necesaria por la asimetría de la densidad. Pase el cursor sobre una caja para leer los cuartiles y sobre un punto para identificar el cantón.

### Facetas: un gráfico por categoría

Las **facetas** son la forma de plotly express de producir varios gráficos en una figura: `facet_col` (o `facet_row`) dibuja un gráfico por cada valor de una columna, con los mismos ejes, y `facet_col_wrap` indica cuántos por fila. Los histogramas de la población de los distritos, por provincia:

In [10]:
# Un histograma por provincia
fig = px.histogram(
    distritos,
    x="poblacion",
    facet_col="provincia",
    facet_col_wrap=4,
    nbins=30,
    title="Distribución de la población de los distritos, por provincia, 2022",
    labels={"poblacion": "Habitantes del distrito"},
    height=500
)

# Solo el nombre de la provincia en el título de cada faceta (sin el prefijo "provincia=")
fig.for_each_annotation(lambda anotacion: anotacion.update(text=anotacion.text.split("=")[1]))
fig.update_layout(xaxis_tickformat=",", bargap=0.05)

mostrar(fig)

Como los ejes son los mismos en todas las facetas, las distribuciones se comparan de forma directa: San José y Alajuela tienen muchos distritos y colas largas; Guanacaste y Limón, pocos distritos y más pequeños.

## Guardar una figura

Una figura de plotly puede guardarse como página web **interactiva** con `write_html()`. El archivo resultante se abre en cualquier navegador, con las etiquetas emergentes y el acercamiento intactos, y puede publicarse con GitHub Pages, como el sitio de la tarea 1:

In [11]:
# Guardado de la última figura como página web interactiva
fig.write_html("distritos-poblacion-provincias.html")

Para guardar una **imagen** estática (PNG, SVG, PDF) se usa `write_image()`, que requiere el paquete adicional [kaleido](https://github.com/plotly/Kaleido). En Colab y en el ambiente `geopython` no está instalado; la alternativa inmediata es el botón de descarga de la barra de herramientas del gráfico.

## matplotlib o plotly

Las dos bibliotecas producen los mismos tipos de gráficos, y en la tarea 2 puede usarse cualquiera de las dos. La elección depende del **destino** del gráfico, como resume la tabla 1.

<figure style="text-align: center; margin: 20px 0;">
    <figcaption><strong>Tabla 1</strong>. Comparación de matplotlib y plotly. Elaboración propia.</figcaption>
    <table class="table table-bordered table-striped" style="margin: 0 auto;">
    <thead>
        <tr><th></th><th>matplotlib (con <code>plot()</code> de pandas)</th><th>plotly express</th></tr>
    </thead>
    <tbody>
        <tr><td>Resultado</td><td>Imagen estática</td><td>Gráfico interactivo (JavaScript)</td></tr>
        <tr><td>Destino natural</td><td>Documentos, artículos, presentaciones, informes en PDF</td><td>Cuadernos de notas, páginas web, aplicaciones (streamlit), exploración de datos</td></tr>
        <tr><td>Información adicional</td><td>Solo la que cabe en el gráfico (anotaciones)</td><td>Etiquetas emergentes con cualquier columna</td></tr>
        <tr><td>Personalización</td><td>Métodos de <code>Axes</code>: control total, más código</td><td>Argumentos de la función y <code>update_layout()</code>: menos código para lo común</td></tr>
        <tr><td>Guardar</td><td>PNG, SVG, PDF con <code>savefig()</code></td><td>HTML interactivo con <code>write_html()</code>; imágenes con kaleido</td></tr>
        <tr><td>Peso del archivo</td><td>Pequeño (una imagen)</td><td>Mayor: el cuaderno guarda los datos del gráfico</td></tr>
        <tr><td>Con mapas</td><td>geopandas dibuja sobre matplotlib</td><td>plotly tiene mapas propios; folium y leafmap son las bibliotecas de mapas web del curso</td></tr>
    </tbody>
    </table>
</figure>

Los criterios de diseño de la parte III (título, etiquetas con unidades, tipo de gráfico según la pregunta, orden, color con significado) son los mismos en las dos bibliotecas. La interactividad no sustituye un buen diseño: un gráfico que solo se entiende pasando el cursor por encima no funciona como imagen descargada ni impreso.

## Clínica de errores

Los errores típicos de plotly express, provocados a propósito. El mensaje del primero es largo, pero su primera línea dice exactamente qué pasó:

In [12]:
# ValueError: "Value of 'y' is not the name of a column in 'data_frame'"
# (errata en el nombre de la columna; el mensaje lista las columnas que sí existen)
px.bar(provincias, x="provincia", y="Poblacion")

ValueError: Value of 'y' is not the name of a column in 'data_frame'. Expected one of ['codigo_provincia', 'provincia', 'poblacion', 'hombres', 'mujeres', 'viviendas', 'viviendas_ocupadas', 'viviendas_desocupadas', 'promedio_ocupantes', 'area_km2', 'densidad', 'tasa_crecimiento_poblacion', 'tasa_crecimiento_viviendas', 'relacion_hombre_mujer'] but received: Poblacion

In [13]:
# ValueError: el mismo error, por otra causa: provincia es el ÍNDICE del resultado de groupby(),
# no una columna (falta reset_index(), como en la parte II)
poblacion_provincia = cantones.groupby("provincia")["poblacion"].sum()

px.bar(poblacion_provincia, x="provincia", y="poblacion")

ValueError: Value of 'x' is not the name of a column in 'data_frame'. Expected one of ['poblacion'] but received: provincia

El tercer caso no es un error de Python. `fig.show()` es la forma de desplegar una figura que aparece en la documentación de plotly, y funciona en Jupyter y en Colab. Pero en el sitio web del curso (y en otros sitios estáticos) el gráfico no aparece, porque `show()` depende de código que solo existe cuando el cuaderno se está ejecutando. La función `mostrar()` definida al inicio usa `display(fig)`, que guarda el gráfico en el cuaderno en un formato que el sitio, Jupyter y Colab saben desplegar. En sus propios cuadernos puede usar `fig.show()`; en los que se publican en el sitio se usa `mostrar(fig)`.

## Asistentes de IA para generar gráficos interactivos

Todo lo dicho en la parte III aplica: el prompt describe el DataFrame y pide el patrón del curso, y la verificación es visual. Con plotly express hay una comprobación adicional, específica de la interactividad: revisar la **etiqueta emergente** de varios puntos, porque es fácil que el asistente muestre en ella columnas de más (el índice, códigos internos) o de menos (el nombre de la entidad). Un prompt de ejemplo:

> Actúe como asistente de visualización de datos con plotly express **(rol)**. Tengo un DataFrame llamado `cantones` con una fila por cantón de Costa Rica y las columnas `provincia` (texto), `canton` (texto), `poblacion` (entero), `area_km2` (decimal) y `densidad` (decimal, habitantes por km²) **(contexto)**. Escriba el código de un gráfico de dispersión de área contra población, con escala logarítmica en ambos ejes, el nombre del cantón al pasar el cursor, la provincia en el color, título y etiquetas de los ejes en español con unidades **(tarea)**. Use `plotly.express`, con `labels` para las etiquetas, y termine con `mostrar(fig)` en lugar de `fig.show()` **(formato)**.

## Resumen

- plotly produce gráficos **interactivos** (etiquetas emergentes, acercamiento, leyenda activa, descarga), adecuados para explorar datos, para páginas web y para aplicaciones; matplotlib produce imágenes **estáticas**, adecuadas para documentos e informes.
- **plotly express** (`px`) tiene una función por tipo de gráfico (`bar`, `line`, `histogram`, `scatter`, `box`) que recibe el DataFrame y los nombres de las columnas, y retorna una figura con **trazas** (`data`) y **disposición** (`layout`).
- Los argumentos `title`, `labels`, `hover_name`, `hover_data`, `color`, `size`, `text_auto`, `log_x` y `facet_col` cubren la mayoría de las necesidades; `update_layout()` y `update_traces()` ajustan el resto. Con una lista en `y`, las columnas se llaman `variable` y `value`.
- Los **gráficos de caja** comparan la distribución de una variable entre categorías; las **facetas** producen un gráfico por categoría con los mismos ejes.
- `write_html()` guarda una página web interactiva. En los cuadernos del sitio del curso se despliega con `mostrar(fig)`, no con `fig.show()`.
- Los criterios de diseño de la parte III no cambian con la interactividad.

## Ejercicios

Los ejercicios se agrupan según la sección del cuaderno a la que corresponden; se recomienda realizarlos al concluir la sección respectiva. Resuélvalos en este mismo cuaderno (en su copia de Colab o local), después de ejecutar las celdas de las secciones anteriores. Varios incluyen una **celda de verificación** que examina la figura creada: para que funcione, nombre las figuras como se indica en el enunciado (`fig_1`, `fig_2`...) y ejecute la celda tal cual después de resolver el ejercicio (en la versión publicada muestra un recordatorio, porque los ejercicios no están resueltos). Las [soluciones de estos ejercicios](https://gf0657-programacionsig.github.io/2026-ii/soluciones-plotly-graficos-interactivos) se publican después de la clase correspondiente.

### Barras y líneas

1. En una figura `fig_1`, grafique en barras horizontales los diez cantones con mayor **densidad** de población, con el más denso arriba, título, etiquetas en español con unidades, el valor sobre cada barra y la provincia en la etiqueta emergente.

In [14]:
# Celda de verificación del ejercicio 1: ejecútela tal cual
try:
    assert fig_1.data[0].type == "bar", "La figura debe ser de barras"
    assert fig_1.data[0].orientation == "h", "Las barras deben ser horizontales (orientation='h')"
    assert len(fig_1.data[0].x) == 10, "Deben graficarse exactamente 10 cantones"
    assert fig_1.layout.title.text, "Falta el título"
    print("Ejercicio 1: ¡correcto!")
except NameError:
    print("Aún no se ha definido fig_1 (ejercicio 1).")

Aún no se ha definido fig_1 (ejercicio 1).


2. En `fig_2`, grafique en barras **agrupadas** las viviendas ocupadas y desocupadas de cada provincia, ordenadas por total de viviendas, con título, etiquetas y los nombres "Ocupadas" y "Desocupadas" en la leyenda. Oculte una serie con la leyenda y observe cómo cambia la escala.

In [15]:
# Celda de verificación del ejercicio 2: ejecútela tal cual
try:
    assert len(fig_2.data) == 2, "Deben graficarse dos series"
    assert fig_2.layout.barmode == "group", "Las barras deben estar agrupadas (barmode='group'), no apiladas"
    assert {traza.name for traza in fig_2.data} == {"Ocupadas", "Desocupadas"}, "Los nombres de las series no son los pedidos"
    print("Ejercicio 2: ¡correcto!")
except NameError:
    print("Aún no se ha definido fig_2 (ejercicio 2).")

Aún no se ha definido fig_2 (ejercicio 2).


3. En `fig_3`, grafique en líneas con marcadores la tasa de crecimiento anual entre censos (`tasa_crecimiento` de `censos`), con título y etiquetas. ¿Entre cuáles censos fue mayor el crecimiento? Compruébelo con la etiqueta emergente.

In [16]:
# Celda de verificación del ejercicio 3: ejecútela tal cual
try:
    assert fig_3.data[0].type == "scatter" and "lines" in fig_3.data[0].mode, "La figura debe ser de líneas"
    assert "markers" in fig_3.data[0].mode, "Faltan los marcadores (markers=True)"
    assert fig_3.layout.title.text, "Falta el título"
    print("Ejercicio 3: ¡correcto!")
except NameError:
    print("Aún no se ha definido fig_3 (ejercicio 3).")

Aún no se ha definido fig_3 (ejercicio 3).


### Histogramas, dispersión y cajas

4. En `fig_4`, grafique el histograma de la densidad de población de los cantones, con título y etiquetas. Pase el cursor sobre la primera barra: ¿cuántos cantones tienen menos de 100 habitantes por km²? (El intervalo exacto depende de `nbins`; indique el que obtuvo.)

In [17]:
# Celda de verificación del ejercicio 4: ejecútela tal cual
try:
    assert fig_4.data[0].type == "histogram", "La figura debe ser un histograma"
    assert fig_4.layout.xaxis.title.text, "Falta la etiqueta del eje horizontal"
    print("Ejercicio 4: ¡correcto!")
except NameError:
    print("Aún no se ha definido fig_4 (ejercicio 4).")

Aún no se ha definido fig_4 (ejercicio 4).


5. Agregue a `cantones` la columna `porcentaje_desocupadas` (viviendas desocupadas entre el total de viviendas, por 100) y, en `fig_5`, grafique la relación entre la densidad (escala logarítmica) y ese porcentaje, con el nombre del cantón al pasar el cursor, la provincia en el color y la cantidad de viviendas en el tamaño de los puntos. Identifique con el cursor los tres cantones con mayor porcentaje.

In [18]:
# Celda de verificación del ejercicio 5: ejecútela tal cual
if "porcentaje_desocupadas" not in cantones.columns:
    print("Aún no se ha creado la columna porcentaje_desocupadas (ejercicio 5).")
else:
    try:
        assert fig_5.layout.xaxis.type == "log", "El eje horizontal debe estar en escala logarítmica (log_x=True)"
        assert len(fig_5.data) == 7, "Debe haber una traza por provincia (color='provincia')"
        assert fig_5.data[0].hovertext is not None, "Falta el nombre del cantón en la etiqueta emergente (hover_name)"
        print("Ejercicio 5: ¡correcto!")
    except NameError:
        print("Aún no se ha definido fig_5 (ejercicio 5).")

Aún no se ha creado la columna porcentaje_desocupadas (ejercicio 5).


6. En `fig_6`, grafique en cajas la distribución de la **población** de los distritos de cada provincia, con escala logarítmica en el eje vertical, un punto por distrito y el nombre del distrito al pasar el cursor. ¿Cuál provincia tiene la mediana más alta? ¿Cuál es el distrito más poblado del país y en qué provincia está?

In [19]:
# Celda de verificación del ejercicio 6: ejecútela tal cual
try:
    assert fig_6.data[0].type == "box", "La figura debe ser de cajas"
    assert fig_6.layout.yaxis.type == "log", "El eje vertical debe estar en escala logarítmica (log_y=True)"
    print("Ejercicio 6: ¡correcto!")
except NameError:
    print("Aún no se ha definido fig_6 (ejercicio 6).")

Aún no se ha definido fig_6 (ejercicio 6).


### Guardar y predecir

7. Guarde la figura del ejercicio 5 como página web interactiva en un archivo llamado `cantones-desocupadas.html`, descárguelo y ábralo en el navegador.

In [20]:
# Celda de verificación del ejercicio 7: ejecútela tal cual
import os
if os.path.exists("cantones-desocupadas.html"):
    print("Ejercicio 7: ¡correcto! El archivo existe.")
else:
    print("Aún no se ha guardado el archivo cantones-desocupadas.html (ejercicio 7).")

Aún no se ha guardado el archivo cantones-desocupadas.html (ejercicio 7).


8. **Prediga antes de ejecutar**: ¿qué muestra el siguiente código? ¿Cuánto mide la barra de San José? Escriba su predicción en una celda de texto y luego ejecútelo.

```python
fig = px.bar(provincias, x="provincia", y=["hombres", "mujeres"])
mostrar(fig)
```

<details>
<summary>Después de ejecutar, haga clic aquí para ver la explicación</summary>

Sin <code>barmode="group"</code>, plotly express <strong>apila</strong> las series: la barra de San José mide la suma de hombres y mujeres, es decir, la población total (1 601 167), con los dos segmentos coloreados. Es un gráfico válido cuando interesa el total y su composición; para comparar hombres con mujeres se necesitan las barras agrupadas. Además, las provincias aparecen en el orden del archivo, y las etiquetas de la leyenda son los nombres de las columnas.

</details>

### Asistentes de IA y tarea 2

9. Pida a un asistente de IA el código del prompt de ejemplo de la sección de asistentes (área contra población de los cantones) y verifíquelo: lea el código, revise el gráfico contra la lista de errores de diseño de la parte III, compare tres etiquetas emergentes con las filas correspondientes de `cantones` y evalúe si el gráfico responde la pregunta. Documente en una celda de texto el asistente, el prompt, el código y el resultado de las comprobaciones.

10. Elija uno de los tres gráficos que planeó en el último ejercicio de la parte III para su tarea 2 y prográmelo dos veces, con sus datos: una con matplotlib y otra con plotly express. Compare los dos resultados con la tabla 1 y decida, en una celda de texto, cuál biblioteca usará en la tarea y por qué.

## Referencias bibliográficas

Instituto Nacional de Estadística y Censos. (2023). *Resultados Estimación de Población y Vivienda 2022* [Conjunto de datos]. INEC. https://admin.inec.cr/sites/default/files/2023-11/reResultadosEstimacionPoblacionVivienda2022_3.xlsx
\
\
Ministerio de Salud. (2022). *Situación Nacional COVID-19: casos positivos acumulados por cantón al 30 de mayo de 2022* [Conjunto de datos]. Ministerio de Salud de Costa Rica. https://github.com/gf0657-programacionsig/2026-ii/tree/main/datos/ministerio-salud
\
\
Plotly. (s. f.-a). Plotly Express in Python. En *Plotly open source graphing library for Python*. Recuperado el 26 de setiembre de 2026, de https://plotly.com/python/plotly-express/
\
\
Plotly. (s. f.-b). Styling Plotly Express figures in Python. En *Plotly open source graphing library for Python*. Recuperado el 26 de setiembre de 2026, de https://plotly.com/python/styling-plotly-express/
\
\
Plotly. (s. f.-c). *Plotly open source graphing library for Python*. Recuperado el 26 de setiembre de 2026, de https://plotly.com/python/